# ETAP on the Senescence task (SenSeqNet v2) — via the etap-clf package

Runs the **same ETAP pipeline** used for ferroptosis on the senescence dataset, to
demonstrate architecture portability across two tasks. Uses the `etap-clf` package
end to end (embed → train → attention).

### Storage strategy (important)
- The **trained model** is saved to **Google Drive** — and re-saved every time
  validation improves — so once training finishes it is permanent and never retrains.
- The **embedding cache** (tens of GB of per-residue vectors) stays on Colab's
  **local disk**. HDF5 does not work reliably on the Drive FUSE mount, and the cache
  is far too large for Drive anyway. Trade-off: if the runtime disconnects *before
  training finishes*, embedding re-runs next session (~1 h). Once training completes,
  you're done for good.
- FASTAs live on Drive (small, read once). Every step skips work already present.

### What it does
1. Reconstruct the 76,053 representative senescence sequences (filter the ~666k
   pre-MMseqs2 FASTA down to the representatives in `sequence_metadata.csv`).
2. `etap --train` — ESM3 embedding (cached to Drive, resumable) + ETAP training,
   random stratified split (seed 42, 64/16/20).
3. Reproduce the test split and `etap --eval --gene-analyze` → per-sequence
   probabilities + attention plots.
4. Figure-ready plots (ROC, PR, confusion, per-gene).

### Requirements
GPU + a **valid** HuggingFace token (ESM3 is gated; accept its license first).
`Data_v2.zip` on your Drive. Embedding ~76k sequences is multi-hour; with Drive
caching it survives disconnects, so a shorter GPU session is fine — just re-run.


## 1 · Install etap-clf (fixed branch) + deps

In [ ]:
# Install from the branch that has the ESM3 dtype fix + resumable embedding.
# (After the PR is merged you can switch @fix/... to the plain main URL.)
!pip install -q --force-reinstall --no-deps \
  "git+https://github.com/Sitgttish/summer26.git@fix/esm3-bfloat16-dtype#subdirectory=eta_package"
# deps (skipped by --no-deps above); install any that are missing
!pip install -q esm biopython h5py scikit-learn openpyxl
print("Install done.")


## 2 · Mount Drive, HuggingFace auth, paths (Drive-backed)

In [ ]:
import os, re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# HuggingFace token for gated ESM3 (Colab secret HF_TOKEN preferred)
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    assert tok and tok.startswith('hf_'), 'HF_TOKEN secret missing or malformed'
    os.environ['HF_TOKEN'] = tok
    print('HF_TOKEN loaded from Colab secret.')
except Exception as e:
    print('WARNING:', e, '\n  Add a valid HF_TOKEN secret (with ESM3 access) or set '
          "os.environ['HF_TOKEN']=... below, or training will fail at login.")

# Paths
PROJECT_ROOT = Path('/content/drive/MyDrive/JR_Ferro')
DATA_ZIP     = PROJECT_ROOT / 'Data_v2.zip'          # adjust if your zip is elsewhere
SEN_DIR      = PROJECT_ROOT / 'senescence'           # ALL persistent outputs live here
SEN_DIR.mkdir(parents=True, exist_ok=True)

LOCAL      = Path('/content/sen'); LOCAL.mkdir(exist_ok=True)   # fast local disk
POS_FASTA  = SEN_DIR / 'sen_pos.fasta'
NEG_FASTA  = SEN_DIR / 'sen_neg.fasta'
TEST_FASTA = SEN_DIR / 'sen_test_labeled.fasta'
CACHE_DIR  = LOCAL / 'sen_cache'    # LOCAL: h5py is unreliable on Drive FUSE, and the
                                    # per-residue cache is tens of GB (too big for Drive)
OUT_DIR    = SEN_DIR / 'sen_out'                     # checkpoint + metrics (persisted to Drive)
RESULTS    = SEN_DIR / 'sen_results'                 # predictions + figures

assert DATA_ZIP.exists(), f'Data_v2.zip not found at {DATA_ZIP} — fix DATA_ZIP.'
print('Zip OK:', DATA_ZIP, '| outputs →', SEN_DIR)


## 3 · Reconstruct the representative pos/neg FASTAs (skipped if present)

The metadata lists the 76,053 representatives; we keep only those records from the
big FASTA, rewrite headers as `>{gene}|label={0/1}` (the format the package's
gene+label parser understands), and split into pos/neg. Written to Drive, so this
runs once.

In [ ]:
def clean_gene(g):
    m = re.match(r'[A-Za-z0-9\-]+', str(g))
    return m.group(0) if m else 'UNK'

if POS_FASTA.exists() and NEG_FASTA.exists():
    npos = sum(1 for _ in open(POS_FASTA) if _.startswith('>'))
    nneg = sum(1 for _ in open(NEG_FASTA) if _.startswith('>'))
    print(f'FASTAs already exist on Drive — pos {npos}  neg {nneg} (skipping rebuild)')
else:
    # extract metadata + big FASTA to local scratch
    with zipfile.ZipFile(DATA_ZIP) as z:
        META_MEMBER  = 'esm_output/esm2t33/sequence_metadata.csv'
        FASTA_MEMBER = 'data_before_MMseqs2/combined_sequences.fasta'
        meta = pd.read_csv(z.open(META_MEMBER))
        combined = LOCAL / 'combined_sequences.fasta'
        if not combined.exists():
            print('Extracting combined FASTA (~350 MB) ...', flush=True)
            with z.open(FASTA_MEMBER) as src, open(combined, 'wb') as dst:
                dst.write(src.read())
    print('Representatives in metadata:', len(meta), '| pos frac:', round(meta['label'].mean(), 3))

    meta['header'] = meta['header'].astype(str).str.strip()
    rep = {h: (clean_gene(g), int(l))
           for h, g, l in zip(meta['header'], meta['gene'], meta['label'])}

    n_pos = n_neg = n_seen = 0
    with open(combined) as fin, open(POS_FASTA, 'w') as fpos, open(NEG_FASTA, 'w') as fneg:
        keep = False; buf = []; dest = None
        for line in fin:
            if line.startswith('>'):
                if keep and buf: dest.write(''.join(buf))
                h = line[1:].strip()
                if h in rep:
                    gene, lab = rep[h]; keep = True; n_seen += 1
                    dest = fpos if lab == 1 else fneg
                    n_pos += (lab == 1); n_neg += (lab == 0)
                    buf = [f'>{gene}|label={lab}\n']
                else:
                    keep = False; buf = []
            elif keep:
                buf.append(line)
        if keep and buf: dest.write(''.join(buf))
    print(f'Matched {n_seen}/{len(rep)} representatives  ->  pos {n_pos}  neg {n_neg}')
    assert n_seen >= 0.98 * len(rep), 'Header match < 98% — inspect meta["header"] vs FASTA.'


## 4 · Train ETAP (embed → train)

Embedding writes to local disk; training then reads it there (reliable, fast). The
best model is saved to **Drive** every time val-AUC improves, so once training
finishes it's permanent. If the runtime dies **before** training completes, re-run
this cell — embedding re-runs, then training. Once a finished model is on Drive,
this cell skips straight past training.

In [ ]:
POS, NEG = str(POS_FASTA), str(NEG_FASTA)
OUTD, CACHE = str(OUT_DIR), str(CACHE_DIR)

def _training_complete(p):
    import torch
    if not p.exists(): return False
    try:
        return 'test_metrics' in torch.load(p, map_location='cpu', weights_only=False)
    except Exception:
        return False

if _training_complete(OUT_DIR / 'best_model.pth'):
    print('Fully-trained model already on Drive — skipping training. Delete '
          f'{OUT_DIR/"best_model.pth"} to retrain.')
else:
    # training writes the best model to Drive every time val-AUC improves, so even
    # an interrupted run leaves the best-so-far checkpoint here.
    cmd = f'etap --train "{POS}" "{NEG}" "{OUTD}/" --cache-dir "{CACHE}" --embed-batch-size 8'
    get_ipython().system(cmd)

import os
print('sen_out:', os.listdir(OUT_DIR) if OUT_DIR.exists() else '(none)')


## 5 · Reproduce the package's exact test split

`run_training` assembles `pos_records + neg_records` and splits with
`train_test_split(test_size=0.20, stratify=y, random_state=42)`. We replicate it with
the package's own `parse_fasta` so indices match, and write the test sequences to a
labeled FASTA.

In [ ]:
from etap.data import parse_fasta
from sklearn.model_selection import train_test_split

pos_rec = parse_fasta(str(POS_FASTA))
neg_rec = parse_fasta(str(NEG_FASTA))
all_rec = pos_rec + neg_rec
y   = np.array([1]*len(pos_rec) + [0]*len(neg_rec))
idx = np.arange(len(y))
idx_tv, idx_test = train_test_split(idx, test_size=0.20, stratify=y, random_state=42)
print(f'Reproduced split — total {len(y)}  test {len(idx_test)}  '
      f'test pos-frac {y[idx_test].mean():.3f}')

with open(TEST_FASTA, 'w') as f:
    for i in idx_test:
        header, gene, seq, _ = all_rec[i]
        f.write(f'>{gene}|label={int(y[i])}\n{seq}\n')
print('Wrote', TEST_FASTA)


## 6 · Evaluate on the test split (+ attention analysis)

In [ ]:
assert (OUT_DIR / 'best_model.pth').exists(), (
    'best_model.pth missing — training (cell 4) has not finished yet. '
    'Re-run cell 4 until it prints a checkpoint path.')
RESULTS.mkdir(parents=True, exist_ok=True)
CKPT = str(OUT_DIR / 'best_model.pth')
PREDS = str(RESULTS / 'predictions.csv')
ADIR  = str(RESULTS / 'analysis')
cmd = f'etap --eval "{CKPT}" "{str(TEST_FASTA)}" "{PREDS}" --gene-analyze --analyze-dir "{ADIR}/"'
get_ipython().system(cmd)

pred = pd.read_csv(PREDS)
print('columns:', list(pred.columns), '| rows:', len(pred))
print(pred.head())


## 7 · Figure-ready plots (ROC, PR, confusion, per-gene)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, precision_recall_curve,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, accuracy_score)

df = pd.read_csv(str(RESULTS / 'predictions.csv'))
y_true = df['true_label'].to_numpy(); y_prob = df['prob_positive'].to_numpy()
y_pred = df['predicted_label'].to_numpy()
auroc = roc_auc_score(y_true, y_prob); ap = average_precision_score(y_true, y_prob)
print(f'Senescence test — Acc {accuracy_score(y_true, y_pred):.4f}  '
      f'AUROC {auroc:.4f}  AP {ap:.4f}')

fpr, tpr, _ = roc_curve(y_true, y_prob)
prec, rec, _ = precision_recall_curve(y_true, y_prob)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(fpr, tpr, lw=2, label=f'AUROC = {auroc:.3f}'); ax[0].plot([0,1],[0,1],'--',color='gray',lw=1)
ax[0].set(xlabel='False positive rate', ylabel='True positive rate', title='Senescence ROC'); ax[0].legend(loc='lower right')
ax[1].plot(rec, prec, lw=2, color='darkorange', label=f'AP = {ap:.3f}')
ax[1].axhline(y_true.mean(), ls='--', color='gray', lw=1, label=f'baseline = {y_true.mean():.2f}')
ax[1].set(xlabel='Recall', ylabel='Precision', title='Senescence PR'); ax[1].legend(loc='lower left')
fig.tight_layout(); fig.savefig(RESULTS / 'sen_roc_pr.png', dpi=200); plt.show()

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.2, 3.8)); ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, f'{v}', ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
ax.set(xticks=[0,1], yticks=[0,1], xticklabels=['neg','pos'], yticklabels=['neg','pos'],
       xlabel='Predicted', ylabel='True', title='Senescence confusion')
fig.tight_layout(); fig.savefig(RESULTS / 'sen_confusion.png', dpi=200); plt.show()

pg = (df.groupby('gene')
        .apply(lambda x: pd.Series({'n': len(x), 'label': int(x['true_label'].iloc[0]),
                                    'acc': (x['predicted_label'] == x['true_label']).mean()}))
        .reset_index())
pg.to_csv(RESULTS / 'sen_per_gene.csv', index=False)
posg, negg = pg[pg.label == 1], pg[pg.label == 0]
print(f'Per-gene: positive macro-acc {posg["acc"].mean():.3f} (n={len(posg)}), '
      f'negative macro-acc {negg["acc"].mean():.3f} (n={len(negg)})')
print('Saved figures + tables to', RESULTS)
